# MSM 데이터 보충 수집 — Notebook 1

**목적:** 보유 자산 외 추가 필요 데이터 수집 (VKOSPI 10년 + 상폐 종목)

**기간:** 2017-01-01 ~ 2026-06-XX

**저장:** `choonsimi-msm/data/`

## 진행 순서
1. 환경 setup
2. **시험 셀 3개** — 결과 확인 후 본수집
   - 시험 A: 상폐 종목 목록 수집 가능 여부
   - 시험 B: 상폐 종목 과거 OHLCV 회수 가능 여부
   - 시험 C: VKOSPI 2017년부터 데이터 존재 여부
3. 본수집 (시험 통과 항목만)
4. 검증 리포트

---
## 1. 환경 Setup

In [ ]:
# Colab 환경 setup
!pip install -q pykrx pyarrow

In [ ]:
import os
import json
import time
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import numpy as np
from pykrx import stock as krx

# 기간
START = "20170101"
END   = datetime.now().strftime("%Y%m%d")
print(f"수집 기간: {START} ~ {END}")

# 저장 루트 (choonsimi-msm clone 후 그 안으로 cd 권장)
# 예: !git clone https://github.com/stanleyim/choonsimi-msm.git
#     %cd choonsimi-msm
ROOT = Path("data")
(ROOT / "raw" / "delisted" / "ohlcv").mkdir(parents=True, exist_ok=True)
(ROOT / "raw" / "delisted" / "flow").mkdir(parents=True, exist_ok=True)
(ROOT / "macro").mkdir(parents=True, exist_ok=True)
(ROOT / "universe").mkdir(parents=True, exist_ok=True)
(ROOT / "state").mkdir(parents=True, exist_ok=True)
(ROOT / "checks").mkdir(parents=True, exist_ok=True)

REPORT = {"start": START, "end": END, "tests": {}, "collection": {}}
print("디렉토리 준비 완료")

---
## 2. 시험 셀

**원칙: 추측 금지. 각 시험 결과 확인 후 본수집 진행.**

### 시험 A — 상폐 종목 목록 수집

In [ ]:
# pykrx에 상폐 종목 list 함수가 있는지 확인
# 후보 1: get_stock_ticker_list with prev=True
# 후보 2: get_market_ticker_list 의 historical date

test_a = {"method_tried": [], "result": None}

# 방법 1: 과거 시점 기준 ticker list - 현재 list와 차집합
try:
    today_kospi  = set(krx.get_market_ticker_list(END, market="KOSPI"))
    today_kosdaq = set(krx.get_market_ticker_list(END, market="KOSDAQ"))
    today_all = today_kospi | today_kosdaq
    
    past_kospi  = set(krx.get_market_ticker_list(START, market="KOSPI"))
    past_kosdaq = set(krx.get_market_ticker_list(START, market="KOSDAQ"))
    past_all = past_kospi | past_kosdaq
    
    delisted_candidates = past_all - today_all
    test_a["method_tried"].append("ticker_list_diff")
    test_a["past_count"] = len(past_all)
    test_a["today_count"] = len(today_all)
    test_a["delisted_candidate_count"] = len(delisted_candidates)
    test_a["sample_delisted"] = sorted(list(delisted_candidates))[:20]
    test_a["result"] = "OK" if len(delisted_candidates) > 0 else "EMPTY"
    print(f"방법1 (ticker_list_diff): {START} 시점 전체 {len(past_all)}, 오늘 {len(today_all)}")
    print(f"→ 상폐 후보: {len(delisted_candidates)}개")
    print(f"샘플 20: {test_a['sample_delisted']}")
except Exception as e:
    test_a["method_tried"].append(f"ticker_list_diff_FAILED: {e}")
    test_a["result"] = "FAIL"
    print(f"방법1 실패: {e}")

REPORT["tests"]["A_delisted_list"] = test_a
print()
print("=== 시험 A 결과 ===")
print(json.dumps(test_a, ensure_ascii=False, indent=2))

### 시험 B — 상폐 종목 OHLCV 회수

In [ ]:
# 시험 A에서 얻은 상폐 후보 중 5개로 OHLCV 회수 시험

test_b = {"samples": [], "result": None}

if test_a.get("result") == "OK" and test_a.get("sample_delisted"):
    samples = test_a["sample_delisted"][:5]
    
    success_count = 0
    for code in samples:
        try:
            df = krx.get_market_ohlcv(START, END, code, adjusted=True)
            rows = len(df)
            first_date = df.index.min().strftime("%Y-%m-%d") if rows > 0 else None
            last_date  = df.index.max().strftime("%Y-%m-%d") if rows > 0 else None
            test_b["samples"].append({
                "code": code, "rows": rows,
                "first": first_date, "last": last_date,
                "status": "OK" if rows > 0 else "EMPTY"
            })
            if rows > 0:
                success_count += 1
            print(f"{code}: {rows}행 ({first_date} ~ {last_date})")
            time.sleep(0.2)
        except Exception as e:
            test_b["samples"].append({"code": code, "status": f"FAIL: {e}"})
            print(f"{code}: 실패 — {e}")
    
    test_b["success_rate"] = success_count / len(samples)
    test_b["result"] = "OK" if success_count >= 3 else "FAIL"
else:
    test_b["result"] = "SKIP (시험 A 실패)"

REPORT["tests"]["B_delisted_ohlcv"] = test_b
print()
print("=== 시험 B 결과 ===")
print(json.dumps(test_b, ensure_ascii=False, indent=2))

### 시험 C — VKOSPI 2017년부터 데이터 존재 여부

In [ ]:
# pykrx VKOSPI = '1330', name_display=False 필수 (DATA_COLLECTION_MANUAL §6.2)

test_c = {"result": None}

try:
    df = krx.get_index_ohlcv(START, END, "1330", name_display=False)
    rows = len(df)
    first = df.index.min().strftime("%Y-%m-%d") if rows > 0 else None
    last  = df.index.max().strftime("%Y-%m-%d") if rows > 0 else None
    test_c["rows"] = rows
    test_c["first"] = first
    test_c["last"] = last
    test_c["columns"] = list(df.columns)
    test_c["result"] = "OK" if rows > 1000 else "INSUFFICIENT"
    print(f"VKOSPI: {rows}행 ({first} ~ {last})")
    print(f"컬럼: {list(df.columns)}")
    print(df.head(3))
    print(df.tail(3))
except Exception as e:
    test_c["result"] = f"FAIL: {e}"
    print(f"실패: {e}")

REPORT["tests"]["C_vkospi_10y"] = test_c
print()
print("=== 시험 C 결과 ===")
print(json.dumps(test_c, ensure_ascii=False, indent=2))

---
## 3. 시험 종합 — 본수집 진행 여부 결정

**시험 결과 확인 후, 형에게 보고 → OK 받고 다음 셀 실행.**

In [ ]:
print("=" * 60)
print("시험 종합")
print("=" * 60)
print(f"A (상폐 목록):  {REPORT['tests']['A_delisted_list']['result']}")
print(f"B (상폐 OHLCV): {REPORT['tests']['B_delisted_ohlcv']['result']}")
print(f"C (VKOSPI 10y): {REPORT['tests']['C_vkospi_10y']['result']}")
print()
print("이 결과를 형에게 보고 후, 본수집 진행.")

# 중간 리포트 저장
with open(ROOT / "checks" / "notebook1_test_report.json", "w", encoding="utf-8-sig") as f:
    json.dump(REPORT, f, ensure_ascii=False, indent=2, default=str)
print("\n중간 리포트 저장: data/checks/notebook1_test_report.json")

---
## 4. 본수집 — VKOSPI 10년

**조건: 시험 C 통과 시.**

In [ ]:
if REPORT["tests"]["C_vkospi_10y"]["result"] == "OK":
    df_vk = krx.get_index_ohlcv(START, END, "1330", name_display=False)
    df_vk.index.name = "date"
    df_vk = df_vk.reset_index()
    
    # 컬럼 정규화 (한글 → 영문)
    rename = {"시가": "open", "고가": "high", "저가": "low",
              "종가": "close", "거래량": "volume", "거래대금": "trade_value"}
    df_vk = df_vk.rename(columns={k: v for k, v in rename.items() if k in df_vk.columns})
    
    out_path = ROOT / "raw" / "vkospi_10y.parquet"
    df_vk.to_parquet(out_path, index=False)
    
    REPORT["collection"]["vkospi"] = {
        "rows": len(df_vk),
        "first": df_vk["date"].min().strftime("%Y-%m-%d"),
        "last":  df_vk["date"].max().strftime("%Y-%m-%d"),
        "path":  str(out_path),
    }
    print(f"VKOSPI 저장: {out_path} ({len(df_vk)}행)")
else:
    print("시험 C 실패 — 수동 다운로드 필요 (형이 제공)")

---
## 5. 본수집 — 상폐 종목 OHLCV + Flow (연도별 parquet)

**조건: 시험 A·B 통과 시.**

**예상 시간:** 상폐 종목 수 × 종목당 약 3초

In [ ]:
if REPORT["tests"]["A_delisted_list"]["result"] == "OK" and REPORT["tests"]["B_delisted_ohlcv"]["result"] == "OK":
    delisted_codes = sorted(test_a["sample_delisted"] + 
                            list(set(krx.get_market_ticker_list(START, market="KOSPI")) |
                                 set(krx.get_market_ticker_list(START, market="KOSDAQ")) -
                                 set(krx.get_market_ticker_list(END, market="KOSPI")) -
                                 set(krx.get_market_ticker_list(END, market="KOSDAQ"))))
    # 중복 제거
    delisted_codes = sorted(set(delisted_codes))
    print(f"상폐 종목 총 {len(delisted_codes)}개 수집 시작")
    
    ohlcv_rows = []
    flow_rows = []
    fail = []
    
    for i, code in enumerate(delisted_codes, 1):
        try:
            # OHLCV adjusted
            df = krx.get_market_ohlcv(START, END, code, adjusted=True)
            if len(df) > 0:
                df = df.reset_index().rename(columns={
                    "날짜": "date", "시가": "open", "고가": "high", "저가": "low",
                    "종가": "close", "거래량": "volume", "등락률": "change_rate"
                })
                df["code"] = code.zfill(6)
                ohlcv_rows.append(df)
            time.sleep(0.15)
            
            # Flow (투자자별)
            try:
                df_flow = krx.get_market_trading_value_by_date(START, END, code)
                if len(df_flow) > 0:
                    df_flow = df_flow.reset_index()
                    df_flow["code"] = code.zfill(6)
                    flow_rows.append(df_flow)
                time.sleep(0.15)
            except Exception as e:
                pass  # flow는 일부 종목에서 실패 가능 — 무시
        except Exception as e:
            fail.append((code, str(e)))
        
        if i % 20 == 0:
            print(f"  진행: {i}/{len(delisted_codes)} ({i/len(delisted_codes)*100:.1f}%)")
    
    # OHLCV 저장 (연도별 partition)
    if ohlcv_rows:
        df_all = pd.concat(ohlcv_rows, ignore_index=True)
        df_all["date"] = pd.to_datetime(df_all["date"])
        df_all["year"] = df_all["date"].dt.year
        for year, df_y in df_all.groupby("year"):
            out = ROOT / "raw" / "delisted" / "ohlcv" / f"year={year}.parquet"
            df_y.drop(columns=["year"]).to_parquet(out, index=False)
        print(f"OHLCV 저장: {len(df_all)}행, {df_all['code'].nunique()}종목")
    
    # Flow 저장
    if flow_rows:
        df_all_flow = pd.concat(flow_rows, ignore_index=True)
        df_all_flow["date"] = pd.to_datetime(df_all_flow["날짜"] if "날짜" in df_all_flow.columns else df_all_flow["date"])
        df_all_flow["year"] = df_all_flow["date"].dt.year
        for year, df_y in df_all_flow.groupby("year"):
            out = ROOT / "raw" / "delisted" / "flow" / f"year={year}.parquet"
            df_y.drop(columns=["year"]).to_parquet(out, index=False)
        print(f"Flow 저장: {len(df_all_flow)}행")
    
    REPORT["collection"]["delisted"] = {
        "total_codes": len(delisted_codes),
        "ohlcv_codes": df_all["code"].nunique() if ohlcv_rows else 0,
        "flow_codes":  df_all_flow["code"].nunique() if flow_rows else 0,
        "failed_count": len(fail),
    }
    print(f"실패: {len(fail)}개")
else:
    print("시험 A 또는 B 실패 — 본수집 스킵")

---
## 6. 최종 리포트 저장

In [ ]:
REPORT["finished_at"] = datetime.now().isoformat()
with open(ROOT / "checks" / "notebook1_report.json", "w", encoding="utf-8-sig") as f:
    json.dump(REPORT, f, ensure_ascii=False, indent=2, default=str)

print("=" * 60)
print("Notebook 1 완료")
print("=" * 60)
print(json.dumps(REPORT, ensure_ascii=False, indent=2, default=str))